# LongCoT task benchmark for Fleet

Compare one native direct LM call and strict Fleet RLM on five easy tasks, one per domain. The pilot is capped at ten serial task attempts and is diagnostic, not statistical evidence or a leaderboard reproduction. Set `COMPARISON_PILOT=False` for the existing single-Fleet integration.

Install notebook tooling once with `uv sync --locked --group benchmarks`, and select this repository's `.venv` kernel. Task preparation downloads public data; live execution and Fleet judges have separate disabled gates.

LongCoT's HF export omits structured verifier payloads for some domains. Join selected HF tasks to the pinned upstream verification package by question ID, requiring identical prompts. Gold and verifier payloads stay in host-side receipts. The published prompt is used verbatim; no benchmark guidance is injected. Historical runs under `outputs/longcot/` remain separate.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "pyproject.toml").is_file(), "Open this notebook from the Fleet repository"
sys.path.insert(0, str(ROOT / "scripts"))

In [ ]:
import hashlib
import json

import pandas as pd
from benchmark_tasks import (
    evaluate_saved_tasks,
    freeze_tasks,
    load_tasks,
    run_comparison_pilot,
    run_tasks,
    score_comparison_pilot,
    score_tasks,
)
from datasets import load_dataset

from fleet_rlm.config import get_settings

MLFLOW_MODE = "local"  # Or "databricks", matching the running Fleet service.
settings = get_settings(MLFLOW_MODE)
BASE_URL = f"http://{settings.host}:{settings.port}"
ALLOW_LIVE = False
ALLOW_JUDGES = False
ALLOW_OFFICIAL_EVALUATION = False  # Local MLflow writes, no model judges.
HF_PIN_VERIFIED = False  # Set only after verifying REVISION through Hugging Face MCP.

In [ ]:
DATASET = "LongHorizonReasoning/longcot"
REVISION = "7f9948570f5239f0f3697c6c9793b4866860c681"
SPLIT = "easy"
COMPARISON_PILOT = True
PER_DOMAIN = 1 if COMPARISON_PILOT else 2
SEED = 20261021
BUNDLE = ROOT / (
    "outputs/benchmarks/longcot-easy-pilot-v1" if COMPARISON_PILOT else "outputs/benchmarks/longcot-easy-v1"
)
RUN_DIR = BUNDLE / ("runs/run-002-direct-rlm" if COMPARISON_PILOT else "runs/run-001")  # Preserve earlier evidence.
if COMPARISON_PILOT:
    settings = settings.with_explicit_overrides(
        fleet_shared_notes_subpath="",
        fleet_root_cache=False,
        fleet_sub_cache=False,
        fleet_sub_model=settings.fleet_root_model,
    )

In [ ]:
from dataclasses import asdict
from importlib.metadata import distribution

import longcot

VERIFIER_COMMIT = "fb9649423f15f5b0091f8e988b100596cac592ca"
DOMAINS = ("logic", "cs", "chemistry", "chess", "math")


def require_verifier(expected=VERIFIER_COMMIT):
    """Require the installed LongCoT Git commit to match ``expected``.

    Raise ValueError on a mismatch; package lookup and metadata decoding errors propagate.
    """
    source = json.loads(distribution("longcot").read_text("direct_url.json") or "{}")
    if source.get("vcs_info", {}).get("commit_id") != expected:
        raise ValueError("Installed longcot differs from the required commit")


def select_longcot(rows, per_domain=2, seed=20261021):
    """Choose a deterministic, seeded sample from each domain.

    Request ``per_domain`` rows per domain, returning them in DOMAINS order.
    Raise ValueError if any domain has fewer rows than requested.
    """
    chosen = []
    for domain in DOMAINS:
        pool = [row for row in rows if row["domain"] == domain]
        pool.sort(key=lambda row: hashlib.sha256(f"{seed}:{row['question_id']}".encode()).hexdigest())
        if len(pool) < per_domain:
            raise ValueError(f"Insufficient {domain} tasks; refusing to change the sample")
        chosen.extend(pool[:per_domain])
    return chosen


def longcot_task(row, canonical):
    """Build a task with the canonical verifier payload and any host-only gold answer.

    ``canonical`` maps question IDs to LongCoT questions. Raise ValueError if prompt, domain,
    or difficulty differs from the HF row; a missing canonical question raises KeyError.
    """
    question = canonical[row["question_id"]]
    if (question.prompt, question.domain, question.difficulty) != (row["prompt"], row["domain"], row["difficulty"]):
        raise ValueError("HF task differs from the pinned verifier's canonical question")
    payload = asdict(question)
    return {
        "case_id": row["question_id"],
        "task": row["prompt"],
        "question": payload,
        "source_group": f"{row['domain']}:{row['template']}",
        "expected_answer": json.dumps(question.answer) if question.answer is not None else None,
    }


def official_score(case, answer):
    """Return 1.0 for a nonblank answer accepted by the pinned verifier, otherwise 0.0.

    Disable math and chemistry model fallbacks. Verifier pin and verification errors propagate.
    """
    require_verifier(bundle["metadata"]["verifier_commit"])
    options = longcot.VerifyOptions(
        math=longcot.MathVerifyOptions(enable_fallback=False),
        chemistry=longcot.ChemistryVerifyOptions(enable_fallback=False),
    )
    return float(bool(answer.strip()) and longcot.verify(longcot.Question(**case["question"]), answer, options=options))

In [ ]:
if BUNDLE.exists():
    bundle = load_tasks(BUNDLE)
else:
    if not HF_PIN_VERIFIED:
        raise ValueError("Verify the pinned Hugging Face dataset through MCP before downloading")
    require_verifier()
    source = load_dataset(DATASET, "all", split=SPLIT, revision=REVISION)
    rows = select_longcot(list(source), PER_DOMAIN, SEED)
    canonical = {question.question_id: question for question in longcot.load_questions(difficulty=SPLIT)}
    bundle = freeze_tasks(
        BUNDLE,
        [longcot_task(row, canonical) for row in rows],
        {
            "dataset": DATASET,
            "dataset_revision": REVISION,
            "split": SPLIT,
            "dataset_url": f"https://huggingface.co/datasets/{DATASET}/tree/{REVISION}",
            "seed": SEED,
            "per_domain": PER_DOMAIN,
            "verifier_commit": VERIFIER_COMMIT,
            "hf_pin_verified": HF_PIN_VERIFIED,
            "protocol": "Published prompts, standard Fleet settings, no verifier model fallback",
        },
    )
display(pd.DataFrame([{"case_id": case["case_id"], "source_group": case["source_group"]} for case in bundle["cases"]]))
print("Frozen task digest:", bundle["digest"])

Verify the pinned dataset through Hugging Face MCP before enabling preparation. Start the isolated local service using `uv run fleet web`, with `FLEET_SHARED_NOTES_SUBPATH=''`, `FLEET_ROOT_CACHE=false`, `FLEET_SUB_CACHE=false`, and `FLEET_SUB_MODEL` equal to the configured root model. Pass these environment overrides only to this launch; do not edit `.env` or TOML. Preserve all configured per-role token limits, retries, and Fleet call budgets. Notebook settings and the service must match.

Set `ALLOW_LIVE=True` only for the authorized pilot. Each arm uses fresh sessions. Domain order is CS, chemistry, chess, math, logic; the first pair runs RLM then direct, alternating first arm thereafter. The first pair's traces, strict route, cleanup, and official verifier results must validate before continuing (wrong answers are valid observations). There is no benchmark generation deadline or Fleet HTTP read timeout (`timeout_seconds=0` disables Fleet's deadline), but native DSPy has a separate 600-second idle-read timeout that its supported LM API cannot disable. The manifest records it. Provider-side limits, output caps, retries, and Fleet call budgets still apply. Raw evidence is saved; uncertainty or unresolved cleanup stops execution. Completed receipts are reused, never replayed. A new protocol needs a new, separately authorized run directory. This is not equal aggregate compute: Fleet can make multiple LM calls. Truncation is known only where the provider reports a finish reason.

In [ ]:
if ALLOW_LIVE:
    if COMPARISON_PILOT:
        run_comparison_pilot(
            BUNDLE, RUN_DIR, settings, base_url=BASE_URL, allow_live=True, official_score=official_score
        )
    else:
        run_tasks(
            BUNDLE, RUN_DIR, base_url=BASE_URL, expected_destination=settings.mlflow_descriptor(), allow_live=True
        )
else:
    print("Live execution disabled; frozen task preparation and saved-output scoring are separate.")

Score existing receipts independently of execution. Official verification below is local, with model fallback disabled. Completed-case accuracy excludes operational failures; all-selected-case accuracy includes failures and unattempted cases in its denominator. Partial results are marked incomplete.

In [ ]:
if COMPARISON_PILOT and (RUN_DIR / "comparison.json").exists():
    report = score_comparison_pilot(BUNDLE, RUN_DIR, official_score)
    display(
        pd.DataFrame(
            {
                arm: {key: value for key, value in summary.items() if key != "cases"}
                for arm, summary in report["arms"].items()
                if summary
            }
        ).T
    )
    display(pd.DataFrame(report["paired_vs_direct"]).T)
    display(
        pd.json_normalize(
            [{"arm": arm, **row} for arm, summary in report["arms"].items() if summary for row in summary["cases"]]
        )
    )
elif (RUN_DIR / "run.json").exists():
    report = score_tasks(BUNDLE, RUN_DIR, official_score)
    display(pd.Series({key: value for key, value in report.items() if key != "cases"}))
    display(pd.json_normalize(report["cases"]))
else:
    print("No saved run yet.")

For the pilot, enable only `ALLOW_OFFICIAL_EVALUATION`: native MLflow dataset discovery/creation, a source-controlled deterministic `@scorer`, and `mlflow.genai.evaluate` run on saved traces. No model judges are built or called. Default MLflow security settings prohibit custom scorer registration; this path does not enable remote executable scorers. The experiment uses the configured evaluation name plus `-official-benchmark` to keep its scorer suite separate. Trace labels exclude benchmarks from optimization curation. Missing outcomes and verifier errors remain explicit. The ordinary single-Fleet path still requires the separate paid `ALLOW_JUDGES` gate. Re-scoring never executes tasks again.

In [ ]:
if COMPARISON_PILOT and ALLOW_OFFICIAL_EVALUATION:
    for arm in json.loads((RUN_DIR / "comparison.json").read_text()).get("arms", ["direct", "hybrid", "rlm"]):
        path = RUN_DIR / arm
        if (path / "scores.json").exists() and json.loads((path / "scores.json").read_text())["scored"]:
            evaluation = evaluate_saved_tasks(BUNDLE, path, settings, official_only=True)
            print(arm, "MLflow evaluation run:", evaluation.run_id)
            display(evaluation.result_df)
elif not COMPARISON_PILOT and ALLOW_JUDGES:
    evaluation = evaluate_saved_tasks(BUNDLE, RUN_DIR, settings, allow_judges=True)
    print("MLflow evaluation run:", evaluation.run_id)
    display(evaluation.result_df)
else:
    print("Fleet judge evaluation disabled.")